In [1]:
pip install fastapi uvicorn openai pdfplumber python-docx pinecone-client


     ---------------------------------------- 0.0/42.8 kB ? eta -:--:--
     -------------------------------------- - 41.0/42.8 kB ? eta -:--:--
     -------------------------------------- 42.8/42.8 kB 416.8 kB/s eta 0:00:00
     ---------------------------------------- 0.0/48.5 kB ? eta -:--:--
     -------- ------------------------------- 10.2/48.5 kB ? eta -:--:--
     -------------------------------- ----- 41.0/48.5 kB 326.8 kB/s eta 0:00:01
     -------------------------------------- 48.5/48.5 kB 245.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/945.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/945.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/945.2 kB ? eta -:--:--
   ---------------------------------------- 10.2/945.2 kB ? eta -:--:--
   ---------------------------------------- 10.2/945.2 kB ? eta -:--:--
   - ------------------------------------- 30.7/945.2 kB 163.8 kB/s eta 0:00:06
   - ---------------------------

In [1]:
from fastapi import FastAPI, UploadFile
import pdfplumber, json
from openai import OpenAI

app = FastAPI()
client = OpenAI(api_key="YOUR_API_KEY")

def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file.file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

@app.post("/parse_resume/")
async def parse_resume(file: UploadFile):
    text = extract_text_from_pdf(file)
    
    prompt = f"""
    Extract this resume into JSON with fields:
    name, email, phone, skills[], experience[{{
      company, role, start, end, description
    }}], education[].
    
    Resume text:
    \"\"\"{text}\"\"\"
    """

    resp = client.responses.create(model="gpt-4o-mini", input=prompt)
    return json.loads(resp.output_text)


In [ ]:
import uvicorn

if __name__ == "__main__":
    uvicorn.run("backend_api:app", host="127.0.0.1", port=8000, reload=True)




INFO:     Will watch for changes in these directories: ['C:\\Users\\91815\\Documents\\ai resume screening']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [16308] using StatReload


In [ ]:
uvicorn backend_api:app --reload --port 8000
